# SDS 04 — Frequency Moments, AMS, DGIM & Decaying Windows Reference

Reusable reference notebook for the SDS comprehensive exam.

**Scope:** frequency moments, AMS, sliding-window models, DGIM, and exponential decay.

The notebook favors readable reference implementations. Keep large data in Spark when possible; the pure-Python state objects are included so you can understand and reuse the algorithm mechanics.

## 0. Recognition

- **F0 / unique count** → HyperLogLog
- **frequency of item x** → Count-Min Sketch
- **F2 / surprise / frequency concentration** → AMS
- **number of 1s in last k ≤ N positions** → DGIM
- **old history should fade gradually** → exponential decay
- **exact recent window and state is manageable** → Spark window / stateful aggregation

## 1. Exact frequency moments — sanity-check baseline

In [ ]:
from collections import Counter

def exact_frequency_moment(values, k):
    counts = Counter(values)
    if k == 0:
        return len(counts)
    return sum(c ** k for c in counts.values())

stream = list("ABACABDABBCA")
for k in [0,1,2,3]:
    print(k, exact_frequency_moment(stream, k))

## 2. Spark exact F2 — useful for verification

This is exact and distributed, but it needs a count for every distinct key. Use it as a correctness baseline when the dataset is manageable; it is not a bounded-memory sketch.

In [ ]:
# from pyspark.sql import functions as F
# exact_f2 = (
#     events.groupBy("key").count()
#           .select((F.col("count") * F.col("count")).alias("sq"))
#           .agg(F.sum("sq").alias("F2"))
# )
# exact_f2.show()

## 3. One AMS estimator from a chosen position

In [ ]:
def ams_from_position(values, pos0):
    """pos0 is a 0-based selected stream position."""
    n = len(values)
    x = values[pos0]
    r = sum(1 for v in values[pos0:] if v == x)
    X = n * (2 * r - 1)
    return {"position": pos0, "element": x, "r": r, "estimate": X}

for p in [2,5,10]:
    print(ams_from_position(stream, p))
print("true F2:", exact_frequency_moment(stream, 2))

## 4. Many AMS variables

In [ ]:
import random, statistics

def ams_f2(values, num_variables=100, seed=42):
    rng = random.Random(seed)
    n = len(values)
    estimates = []
    for _ in range(num_variables):
        pos = rng.randrange(n)
        estimates.append(ams_from_position(values, pos)["estimate"])
    return statistics.fmean(estimates), estimates

est, xs = ams_f2(stream, num_variables=5000)
print("AMS estimate:", est)
print("true F2:", exact_frequency_moment(stream, 2))

## 5. General kth-moment AMS estimator

In [ ]:
def ams_k_from_position(values, pos0, k):
    n = len(values)
    x = values[pos0]
    r = sum(1 for v in values[pos0:] if v == x)
    X = n * (r**k - (r-1)**k)
    return X

for k in [2,3,4]:
    vals=[ams_k_from_position(stream,p,k) for p in range(len(stream))]
    print(k, "average over every possible sampled position =", sum(vals)/len(vals),
          "true =", exact_frequency_moment(stream,k))

## 6. Online AMS idea — reservoir-sample the position

For an unknown-length stream, each AMS variable can maintain a uniformly sampled stream position using Reservoir Sampling with k=1. If the reservoir replaces the sampled position, reset the selected item and suffix count.

In [ ]:
class OnlineAMSVariable:
    def __init__(self, seed=None):
        self.rng = random.Random(seed)
        self.n = 0
        self.element = None
        self.r = 0

    def update(self, x):
        self.n += 1
        # Reservoir sampling of one position: replace with probability 1/n
        if self.rng.randrange(self.n) == 0:
            self.element = x
            self.r = 1
        elif x == self.element:
            self.r += 1

    def f2_estimate(self):
        if self.n == 0:
            return 0.0
        return self.n * (2*self.r - 1)

vars_=[OnlineAMSVariable(seed=i) for i in range(1000)]
for x in stream:
    for v in vars_:
        v.update(x)
print("online AMS avg:", statistics.fmean(v.f2_estimate() for v in vars_))

## 7. Spark AMS teaching pattern with a small broadcast sample table

Assume `events(idx, key)` is ordered by a stable integer `idx`. Generate a small set of sampled positions, join them to their selected keys, then let Spark count suffix occurrences. The sample table is tiny; the event table stays distributed.

In [ ]:
# from pyspark.sql import functions as F
# n = events.count()
# # sample_points columns: sample_id, idx0, selected_key
# counts = (
#     events.join(F.broadcast(sample_points),
#                 events.key == sample_points.selected_key)
#           .where(events.idx >= sample_points.idx0)
#           .groupBy("sample_id")
#           .agg(F.count("*").alias("r"))
# )
# estimates = counts.withColumn("X", F.lit(n) * (2*F.col("r") - 1))
# estimates.agg(F.avg("X").alias("F2_hat")).show()

## 8. Window models

In [ ]:
def describe_window(kind):
    return {
        "landmark": "all history since a fixed start",
        "sliding": "only the most recent N items / T time units",
        "decaying": "all history but old values receive progressively smaller weight",
    }[kind]

for k in ["landmark","sliding","decaying"]:
    print(k, "->", describe_window(k))

## 9. Minimal DGIM state

In [ ]:
from dataclasses import dataclass

@dataclass
class Bucket:
    size: int
    end: int   # timestamp of most recent 1 in bucket

class DGIM:
    """Basic DGIM: at most two buckets of each size."""
    def __init__(self, window_size):
        self.N = int(window_size)
        self.t = 0
        self.buckets = []  # newest -> oldest

    def _expire(self):
        cutoff = self.t - self.N
        self.buckets = [b for b in self.buckets if b.end > cutoff]

    def _compress(self):
        while True:
            sizes = sorted(set(b.size for b in self.buckets))
            changed = False
            for size in sizes:
                idx = [i for i,b in enumerate(self.buckets) if b.size == size]
                if len(idx) > 2:
                    # newest->oldest list: merge the two oldest of this size
                    i_newer_old = idx[-2]
                    i_oldest = idx[-1]
                    new_end = self.buckets[i_newer_old].end
                    # remove highest index first
                    self.buckets.pop(i_oldest)
                    self.buckets.pop(i_newer_old)
                    self.buckets.append(Bucket(size*2, new_end))
                    self.buckets.sort(key=lambda b: b.end, reverse=True)
                    changed = True
                    break
            if not changed:
                break

    def update(self, bit):
        self.t += 1
        self._expire()
        if int(bit) == 1:
            self.buckets.insert(0, Bucket(1, self.t))
            self._compress()
        self._expire()

    def query(self, k):
        k = min(int(k), self.N, self.t)
        if k <= 0:
            return 0.0
        boundary = self.t - k + 1
        active = [b for b in self.buckets if b.end >= boundary]
        if not active:
            return 0.0
        # newest -> oldest; all except oldest are full, oldest is half
        return sum(b.size for b in active[:-1]) + active[-1].size/2.0

    def snapshot(self):
        return [(b.size,b.end) for b in self.buckets]


## 10. DGIM sanity check against exact recent counts

In [ ]:
bits = [int(c) for c in "10101100111011011000101110110010110"]
dg = DGIM(window_size=32)
history=[]
for b in bits:
    history.append(b)
    dg.update(b)
print("buckets newest->oldest:", dg.snapshot())
for k in [5,10,15,25,32]:
    exact = sum(history[-k:])
    approx = dg.query(k)
    print(k, "exact=",exact,"DGIM=",approx)

## 11. DGIM update rules — lookup

1. Advance timestamp and expire buckets whose right endpoint is outside the N-window.
2. New 0 → no bucket.
3. New 1 → create size-1 bucket at current timestamp.
4. If 3 buckets have the same size, merge the **two oldest** into one bucket twice the size.
5. Repeat recursively.
6. Query last k → full newer buckets + half oldest overlapping bucket.
7. Basic relative error ≈ at most 50%; more buckets per size reduce error.

## 12. Spark exact sliding-window reference

In [ ]:
# Structured Streaming event-time example
# from pyspark.sql import functions as F
# result = (
#     events.withWatermark("event_time", "10 minutes")
#           .groupBy(F.window("event_time", "10 minutes"), "key")
#           .count()
# )

## 13. DGIM and Spark positioning

DGIM is an **ordered, stateful** algorithm. Its bucket list is small state; the raw N-bit history should not be collected. In a streaming engine, maintain one DGIM state per logical bit stream/key. Exact API details vary by Spark version, so keep the engine-neutral `DGIM` state object above as the algorithm reference and map it to the available stateful Structured Streaming primitive on the exam server.

## 14. Exponential decaying sum

In [ ]:
class DecayedSum:
    def __init__(self, c):
        if not 0 < c < 1:
            raise ValueError("c must be between 0 and 1")
        self.c = float(c)
        self.value = 0.0

    def update(self, x):
        self.value = (1-self.c)*self.value + float(x)
        return self.value

d = DecayedSum(c=0.2)
for x in [1,0,1,1]:
    print(x, d.update(x))

## 15. Choose c from a desired half-life

In [ ]:
import math

def decay_c_from_half_life(h):
    return 1 - 2**(-1.0/h)

def half_life_from_c(c):
    return math.log(0.5) / math.log(1-c)

for h in [5,20,100]:
    c=decay_c_from_half_life(h)
    print("half-life",h,"c=",c,"check=",half_life_from_c(c))

## 16. Lazy per-key exponential decay

In [ ]:
class LazyDecayedCounts:
    def __init__(self, c):
        self.c=float(c)
        self.state={}  # key -> (score,last_t)

    def update(self, key, t, amount=1.0):
        score,last_t=self.state.get(key,(0.0,t))
        delta=t-last_t
        score *= (1-self.c)**delta
        score += amount
        self.state[key]=(score,t)
        return score

    def score_at(self,key,t):
        if key not in self.state: return 0.0
        score,last_t=self.state[key]
        return score*(1-self.c)**(t-last_t)

recent=LazyDecayedCounts(0.1)
for t,key in enumerate(["A","B","A","A","B","C","A"],start=1):
    recent.update(key,t)
print({k:recent.score_at(k,7) for k in recent.state})

## 17. Spark batch reference for recency-weighted counts

If you have a finite DataFrame with an integer time/index and want a decayed score at a chosen current index, Spark can compute the weighted sum directly. For a true endless stream, prefer stateful lazy updates rather than rescanning history.

In [ ]:
# from pyspark.sql import functions as F
# c = 0.01
# current_t = events.agg(F.max("idx")).first()[0]
# decayed = (
#     events.withColumn("weight", F.pow(F.lit(1-c), F.lit(current_t)-F.col("idx")))
#           .groupBy("key")
#           .agg(F.sum("weight").alias("recent_score"))
# )

## 18. Critique template — AMS

Check these points:

1. Did the answer define `F2 = sum_i m_i^2`?
2. Did it choose a **random stream position**, not merely a random distinct key?
3. Is `r` defined as the suffix count of the selected element from that position onward?
4. Is the estimator `X = n(2r-1)`?
5. Does it explain unbiasedness using `1+3+...+(2m-1)=m^2`?
6. Does it use multiple variables to reduce variance?
7. Does the implementation avoid storing the full frequency dictionary if the point is bounded-memory estimation?

## 19. Critique template — DGIM

Check these points:

1. Is the problem specifically a binary recent-window count?
2. Are bucket sizes powers of two?
3. Are there at most two buckets of each size in basic DGIM?
4. Are the **two oldest** same-size buckets merged when a third appears?
5. Does the bucket store its rightmost timestamp?
6. Does the query use all newer full buckets plus **half** the oldest overlapping bucket?
7. Does it state the ~50% basic relative-error bound and logarithmic bucket count?

## 20. Fast formulas

- `F_k = sum_i m_i^k`
- `F0 = distinct count`
- `F1 = n`
- `F2 = sum_i m_i^2`
- AMS F2: `X = n(2r-1)`
- AMS Fk: `X = n[r^k-(r-1)^k]`
- DGIM query: full newer buckets + half oldest overlapping bucket
- Basic DGIM error: about ≤ 50%
- Generalized DGIM: error about ≤ `1/(r-1)`
- Exponential weight: `(1-c)^age`
- Decay recurrence: `S_new = (1-c)S_old + a_new`
- Half-life: `c = 1 - 2^(-1/h)`